Coauthored and disputed - test set, not train
Train - hamilton and madison


In [1]:

# 1) Imports and paths
from pathlib import Path
import pandas as pd
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.dtm import DTM, Vectorizer
from sklearn.svm import SVC
from lexos.classification import trainer
import lexos.corpus.corpus_stats as cs


BASE = Path.cwd()  # run this notebook from doc_src/docs/tutorials/classification
DATA = BASE / "fed_papers"

print("Base:", BASE)
print("Data dir exists:", DATA.exists())

Base: c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\doc_src\docs\tutorials\classification
Data dir exists: True


In [2]:
# 2) Collect train and test files
train_dirs = ["HAMILTON", "MADISON"]
test_dirs = ["COAUTHORED", "DISPUTED"]

train_files = []
for d in train_dirs:
    train_files.extend(sorted((DATA / d).glob("*.txt")))

test_files = []
for d in test_dirs:
    test_files.extend(sorted((DATA / d).glob("*.txt")))

print("Train files:", len(train_files), "Test files:", len(test_files))
assert train_files and test_files, "No files found; check working directory and folder structure."

Train files: 65 Test files: 15


In [3]:
# 3) Scrubber and tokenizer
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

In [ ]:
# 4) Build training corpus (HAMILTON/MADISON)
train_token_lists, train_doc_ids, y_train = [], [], []

for f in train_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer(clean)  # same as tokenizer.make_doc(clean)
    tokens = [t.text for t in doc if t.text.strip()]
    train_token_lists.append(tokens)
    train_doc_ids.append(f.name)       # row label for DTM (not used by sklearn)
    y_train.append(f.parent.name)      # target = HAMILTON or MADISON

print("Train docs:", len(train_token_lists), "Targets:", set(y_train))
assert len(train_token_lists) == len(train_doc_ids) == len(y_train) and len(train_token_lists) > 0

In [ ]:
# 5) Build test corpus (COAUTHORED/DISPUTED) — labels here are group tags, not training classes
test_token_lists, test_doc_ids, test_groups = [], [], []

for f in test_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer(clean)
    tokens = [t.text for t in doc if t.text.strip()]
    test_token_lists.append(tokens)
    test_doc_ids.append(f.name)
    test_groups.append(f.parent.name)  # COAUTHORED or DISPUTED

print("Test docs:", len(test_token_lists), "Groups:", set(test_groups))
assert len(test_token_lists) == len(test_doc_ids) == len(test_groups) and len(test_token_lists) > 0

In [ ]:
# 6) Vectorize: fit on TRAIN only, transform TEST with the same vocabulary
dtm_train = DTM(vectorizer=Vectorizer())
_ = dtm_train(docs=train_token_lists, labels=train_doc_ids)
X_train = dtm_train.doc_term_matrix

# Transform test using the fitted vectorizer
test_strings = [" ".join(toks) for toks in test_token_lists]
X_test = dtm_train.vectorizer.transform(test_strings)

print("Train DTM:", X_train.shape, "Vocab size:", len(dtm_train.sorted_terms_list))
print("Test DTM:", X_test.shape)

### What this cell does

- Trains a classifier on the HAMILTON vs. MADISON training set and then uses the fitted model to label the held-out COAUTHORED / DISPUTED documents.
- Steps:
    1. Fit an SVC (linear kernel) to X_train and y_train using `trainer.fit_classifier`.
    2. Use the returned classifier (`clf`) to predict labels for X_test.
    3. Build `pred_df` with test document IDs, their group (COAUTHORED / DISPUTED), and the predicted author.
    4. Display the predictions and print counts of predicted authors by test set.

### Why use trainer.fit_classifier

- Encapsulation and consistency: `fit_classifier` is a convenience wrapper that centralizes model creation and training. It accepts model selection and hyperparameters (here `model="svc", kernel="linear"`) and returns a fitted estimator with a standard scikit-learn-like API.
- Reduces leakage risk: by providing a single entry point for fitting, it helps ensure training is performed strictly on the training data (no accidental leakage from test data or preprocessing mismatches).
- Reproducibility and defaults: the function can apply consistent defaults (e.g., label encoding, scaling, class-handling, or internal validation) so experiments are comparable and easier to reproduce.
- Ease of use: it returns a fitted object you can immediately call `.predict()` on, which simplifies the notebook workflow and downstream evaluation/inspection (as in `pred_df` and the group counts printed below).

Notes:
- Here we chose a linear SVC; predictions in `pred_df` reflect that model's decision boundary given the train-fit vocabulary and features.

In [ ]:
# 7) Train classifier on HAMILTON/MADISON and predict authorship for COAUTHORED/DISPUTED
clf = trainer.fit_classifier(X_train, y_train, model="svc", kernel="linear") 
# fit_classifier() is a new method that allows to train
# on an already split training set 

y_pred_test = clf.predict(X_test)

pred_df = pd.DataFrame({
    "doc_id": test_doc_ids,
    "set": test_groups,
    "pred_author": y_pred_test
}).sort_values(["set", "doc_id"])

display(pred_df)
print("\nPrediction counts by set:")
print(pred_df.groupby(["set", "pred_author"]).size())

### 8) Add per‑document text statistics (CorpusStats)
Compute WordCount, UniqueWordCount, CharCount, AvgWordLength, TTR, and HapaxLegomenonRate for every document, then join with predictions.

In [ ]:
# Reload CorpusStats if you just edited lexos
import importlib
import lexos.corpus.corpus_stats as cs
importlib.reload(cs)
from lexos.corpus.corpus_stats import CorpusStats

In [ ]:
# Build docs and raw_texts aligned to train/test ordering
all_paths = [*train_files, *test_files]

ids = [p.name for p in all_paths]                # unique id per doc
labels_for_stats = ids                           # index label (display as filename)
sets_list = [p.parent.name for p in all_paths]   # HAMILTON/MADISON/COAUTHORED/DISPUTED

token_lists_for_stats = []
raw_texts_list = []

for p in all_paths:
    raw = p.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer(clean)
    toks = [t.text for t in doc if t.text.strip()]
    token_lists_for_stats.append(toks)
    raw_texts_list.append(raw)  # enables exact CharCount (includes spaces & punctuation)

# docs must be List[Tuple[id, label, tokens]]
docs = list(zip(ids, labels_for_stats, token_lists_for_stats))

stats = CorpusStats(docs=docs, raw_texts=raw_texts_list)
df_stats = stats.doc_stats_df.copy()

# add the set for grouping
df_stats.insert(0, "set", sets_list)

cols = [
    "word_count",
    "unique_word_count",
    "char_count",
    "avg_word_length",
    "ttr",
    "hapax_legomenon_rate",
]

display(df_stats[["set", *cols]])

In [ ]:
# Aggregates by set (mean/median/min/max)
agg = df_stats.groupby("set")[cols].agg(["mean", "median", "min", "max"]).round(3)
display(agg)

In [ ]:
# Join stats with predictions for easy inspection
# pred_df must exist from the previous cell
stats_for_join = (
    df_stats.reset_index()
           .rename(columns={"index": "doc_id", "Documents": "doc_id"})
           .loc[:, ["doc_id", *cols]]
)

pred_with_stats = pred_df.merge(stats_for_join, on="doc_id", how="left")
display(pred_with_stats.sort_values(["set", "doc_id"]))

# Optional: export
# pred_with_stats.to_csv("fedpapers_predictions_with_stats.csv", index=False)